# Cityscapes: полный запуск эксперимента в Google Colab

Notebook только настраивает окружение и вызывает скрипты проекта. Dataset, обучение, метрики и оценка остаются в `src/`. Cityscapes загружается через `kagglehub.dataset_download("electraawais/cityscape-dataset")`. Перед запуском выберите **Runtime → Change runtime type → GPU**.

## 1. Проверка GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU не найден. Выберите Runtime → Change runtime type → GPU.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
!nvidia-smi

## 2. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub

path = kagglehub.dataset_download("electraawais/cityscape-dataset")
print('KaggleHub dataset:', path)

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/atikkhon/diploma.git"
CITYSCAPES_ROOT = Path(path).expanduser().resolve()

PROJECT_DIR = Path('/content/Diploma')
DRIVE_ROOT = Path('/content/drive/MyDrive/cityscapes_robustness')
LOG_DIR = DRIVE_ROOT / 'logs'
STAGE_DIR = DRIVE_ROOT / 'stage_markers'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
TRAINING_METRICS_DIR = DRIVE_ROOT / 'training_metrics'
MLFLOW_DB = DRIVE_ROOT / 'mlflow.db'
MLFLOW_ARTIFACT_DIR = DRIVE_ROOT / 'mlartifacts'
SPLIT_MANIFEST = DRIVE_ROOT / 'split_manifest.csv'
RUNTIME_CONFIG = DRIVE_ROOT / 'experiment_colab.yaml'

for directory in (DRIVE_ROOT, LOG_DIR, STAGE_DIR, CHECKPOINT_DIR, TRAINING_METRICS_DIR, MLFLOW_ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    'REPO_URL': REPO_URL,
    'PROJECT_DIR': str(PROJECT_DIR),
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'LOG_DIR': str(LOG_DIR),
    'STAGE_DIR': str(STAGE_DIR),
    'CHECKPOINT_DIR': str(CHECKPOINT_DIR),
    'SPLIT_MANIFEST': str(SPLIT_MANIFEST),
    'RUNTIME_CONFIG': str(RUNTIME_CONFIG),
    'MLFLOW_TRACKING_URI': f'sqlite:///{MLFLOW_DB}',
    'MLFLOW_ARTIFACT_ROOT': MLFLOW_ARTIFACT_DIR.resolve().as_uri(),
})
print('Drive artifacts:', DRIVE_ROOT)
print('MLflow tracking URI:', os.environ['MLFLOW_TRACKING_URI'])
print('MLflow artifact root:', os.environ['MLFLOW_ARTIFACT_ROOT'])

## 3. Клонирование или обновление Git-репозитория

При повторном запуске выполняется `git pull --ff-only`; локальные изменения в `/content/Diploma` не перезаписываются.

In [ ]:
%%bash
set -euo pipefail
if [[ -d "$PROJECT_DIR/.git" ]]; then
  git -C "$PROJECT_DIR" pull --ff-only 2>&1 | tee -a "$LOG_DIR/git_update.log"
else
  git clone "$REPO_URL" "$PROJECT_DIR" 2>&1 | tee -a "$LOG_DIR/git_update.log"
fi

## 4. Установка `requirements.txt`

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pip install -r requirements.txt 2>&1 | tee -a "$LOG_DIR/install.log"

### Проверка SQLite MLflow

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.tracking import check_mlflow_connection

mlflow_info = check_mlflow_connection('cityscapes_robustness')
print(mlflow_info)

## 5. Пути Cityscapes и Colab-конфигурация

Manifest, checkpoint, история обучения, логи и MLflow переживают разрыв runtime, потому что сохраняются в Google Drive.

In [ ]:
import sys
import yaml

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.dataset import discover_cityscapes_layout, prepare_train_id_masks

layout = discover_cityscapes_layout(CITYSCAPES_ROOT)
# Маски читаются на каждой эпохе, поэтому держим кэш на локальном SSD Colab.
# После обрыва runtime эта ячейка быстро восстановит его из Kaggle labelIds.
prepared_gtfine = Path('/content/prepared_gtFine')
train_masks = prepare_train_id_masks(
    layout['train_masks'], prepared_gtfine / 'train'
)
val_masks = prepare_train_id_masks(
    layout['val_masks'], prepared_gtfine / 'val'
)

with (PROJECT_DIR / 'configs/experiment.yaml').open(encoding='utf-8') as file:
    config = yaml.safe_load(file)
config['data']['root'] = str(CITYSCAPES_ROOT)
config['data']['train_images'] = str(layout['train_images'])
config['data']['train_masks'] = str(train_masks)
config['data']['official_val_images'] = str(layout['val_images'])
config['data']['official_val_masks'] = str(val_masks)
config['data']['split_file'] = str(SPLIT_MANIFEST)
config['training']['checkpoint_dir'] = str(CHECKPOINT_DIR)
config['training']['history_dir'] = str(TRAINING_METRICS_DIR)
config['training']['log_interval'] = 25
config['evaluation']['metrics_dir'] = str(PROJECT_DIR / 'outputs/metrics')
config['evaluation']['predictions_dir'] = str(PROJECT_DIR / 'outputs/predictions')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)
print('Runtime config:', RUNTIME_CONFIG)
print('Train images:', layout['train_images'])
print('Train masks:', train_masks)
print('Official val images:', layout['val_images'])
print('Official val masks:', val_masks)

## 6. Создание `split_manifest.csv`

Manifest пересоздаётся детерминированно, чтобы в нём всегда были актуальные локальные пути. При повторном запуске полная проверка масок не дублируется.

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -s "$SPLIT_MANIFEST" ]]; then
  echo "Обновление путей существующего manifest без повторной полной проверки масок" | tee -a "$LOG_DIR/create_split.log"
  python scripts/create_split.py --config "$RUNTIME_CONFIG" --skip-mask-validation 2>&1 | tee -a "$LOG_DIR/create_split.log"
else
  python scripts/create_split.py --config "$RUNTIME_CONFIG" 2>&1 | tee -a "$LOG_DIR/create_split.log"
fi

## 7. Запуск тестов

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -v 2>&1 | tee -a "$LOG_DIR/pytest.log"

## 8. Dataset smoke test

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/smoke_test_dataset.py --config "$RUNTIME_CONFIG" --split dev 2>&1 | tee -a "$LOG_DIR/dataset_smoke_test.log"

In [ ]:
from IPython.display import Image, display
smoke_png = PROJECT_DIR / 'outputs/figures/dataset_smoke_test.png'
if not smoke_png.is_file():
    raise FileNotFoundError(f'Smoke-test PNG не найден: {smoke_png}')
display(Image(filename=str(smoke_png)))

## Выбор отдельной модульной версии

Первые восемь разделов используют стабильную `main`. Теперь переключите локальный Colab-клон на отдельную ветку U-Net и проверьте её тестами.

In [ ]:
%%bash
set -euo pipefail
git -C "$PROJECT_DIR" fetch origin codex/unet-modular-pipeline
git -C "$PROJECT_DIR" switch codex/unet-modular-pipeline
git -C "$PROJECT_DIR" pull --ff-only
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -q

## 9. Independent U-Net run parameters

Set `RUN_NAME` and U-Net parameters.

Folder layout on Google Drive:

- `runs/<RUN_NAME>/` stores metrics, CSV files, PNG previews, plots, config, and MLflow run id.
- `models/<RUN_NAME>/` stores only model checkpoints: `best.pt` and `last.pt`.

Run modes:

- New training from scratch: `RESUME_TRAINING = False`, `CONTINUE_FROM_RUN = None`.
- Resume after a runtime interruption: old `RUN_NAME`, `RESUME_TRAINING = True`, `CONTINUE_FROM_RUN = None`. Existing `run_config.yaml` is not overwritten.
- Continue training in a new run: new `RUN_NAME`, `RESUME_TRAINING = False`, `CONTINUE_FROM_RUN = 'old_run_name'`. In this mode, `epochs` below means the number of new epochs to add after the selected checkpoint.


In [ ]:
import os
import yaml

SELECTED_MODEL = 'unet'
RUN_NAME = 'unet_experiment_01'
RESUME_TRAINING = False
CONTINUE_FROM_RUN = None  # example: 'unet_experiment_01'
INIT_CHECKPOINT = 'last'  # 'last' for true continuation, 'best' to start from the best checkpoint

if RESUME_TRAINING and CONTINUE_FROM_RUN is not None:
    raise ValueError('Choose only one mode: RESUME_TRAINING or CONTINUE_FROM_RUN')
if INIT_CHECKPOINT not in {'last', 'best'}:
    raise ValueError("INIT_CHECKPOINT must be 'last' or 'best'")

MODEL_SETTINGS = {
    'unet': {
        'model': {
            'name': 'unet',
            'encoder_name': 'resnet34',
            'encoder_weights': 'imagenet',
        },
        'training': {
            'epochs': 8,
            'batch_size': 8,
            'num_workers': 2,
            'log_interval': 25,
            'learning_rate': 0.0003,
            'weight_decay': 0.0001,
            'device': 'auto',
            'mixed_precision': True,
        },
        'evaluation': {
            'batch_size': 8,
        },
    },
}

selected = MODEL_SETTINGS[SELECTED_MODEL]
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
MODEL_DIR = DRIVE_ROOT / 'models' / RUN_NAME
RUN_CONFIG = RUN_DIR / 'run_config.yaml'
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if RESUME_TRAINING and RUN_CONFIG.exists():
    with RUN_CONFIG.open(encoding='utf-8') as file:
        run_config = yaml.safe_load(file)
    print('Resume uses the existing run_config.yaml without overwriting it')
else:
    with RUNTIME_CONFIG.open(encoding='utf-8') as file:
        run_config = yaml.safe_load(file)
    run_config['project_root'] = str(PROJECT_DIR)
    run_config['run'] = {
        'name': RUN_NAME,
        'output_dir': str(RUN_DIR),
        'model_dir': str(MODEL_DIR),
    }
    run_config['model'] = selected['model']
    run_config['training'].update(selected['training'])
    run_config['evaluation'].update(selected['evaluation'])
    run_config['tracking']['experiment_name'] = 'cityscapes_unet'
    for key in ('checkpoint_dir', 'history_dir'):
        run_config['training'].pop(key, None)
    for key in ('metrics_dir', 'figures_dir', 'predictions_dir'):
        run_config['evaluation'].pop(key, None)
    run_config['corruptions'] = {
        'darkness': {
            'levels': {
                1: {'factor': 0.75},
                2: {'factor': 0.55},
                3: {'factor': 0.35},
            }
        }
    }
    with RUN_CONFIG.open('w', encoding='utf-8') as file:
        yaml.safe_dump(run_config, file, allow_unicode=True, sort_keys=False)

os.environ['RUN_CONFIG'] = str(RUN_CONFIG)
os.environ['RUN_NAME'] = RUN_NAME
os.environ['RUN_DIR'] = str(RUN_DIR)
os.environ['MODEL_DIR'] = str(MODEL_DIR)
os.environ['RESUME_TRAINING'] = '1' if RESUME_TRAINING else '0'
os.environ['CONTINUE_FROM_RUN'] = '' if CONTINUE_FROM_RUN is None else str(CONTINUE_FROM_RUN)
os.environ['INIT_CHECKPOINT'] = INIT_CHECKPOINT
print('Run:', RUN_NAME)
print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
print('Config:', RUN_CONFIG)
print('Resume:', RESUME_TRAINING)
print('Continue from run:', CONTINUE_FROM_RUN)
print('Init checkpoint:', INIT_CHECKPOINT)
print('Configured epochs:', run_config['training']['epochs'])


## 10. Train, resume, or continue the selected run


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -n "$CONTINUE_FROM_RUN" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --continue-from-run "$CONTINUE_FROM_RUN" --init-checkpoint "$INIT_CHECKPOINT" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
elif [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print('GPU cache очищен')

## 11. Clean segmentation preview

Set an image index from the official Cityscapes validation split. This is the same split used by clean/darkness evaluation.


In [ ]:
IMAGE_INDEX = 0
os.environ['IMAGE_INDEX'] = str(IMAGE_INDEX)

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition clean 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_clean.log"

In [ ]:
clean_preview = RUN_DIR / 'figures' / f'segmentation_clean_index_{IMAGE_INDEX}.png'
display(Image(filename=str(clean_preview)))

## 12. Clean evaluation выбранного запуска

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"

## 13. Darkness evaluation

Выберите вручную уровень 1, 2 или 3. Каждая оценка добавляется в CSV и создаёт отдельный дочерний run в MLflow.

In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY должен быть 1, 2 или 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"

## 14. Визуализация сегментации после darkness

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"

In [ ]:
darkness_preview = RUN_DIR / 'figures' / f'segmentation_darkness_s{DARKNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(darkness_preview)))

## 15. Saved results

The lightweight `RUN_DIR` contains metrics, CSV files, plots, previews, and config. Checkpoints are stored separately in `MODEL_DIR`.


In [ ]:
import pandas as pd
from IPython.display import Image, display

print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
display(pd.read_csv(RUN_DIR / 'metrics' / 'training_history.csv'))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

for plot_name in [
    'training_loss_curve.png',
    'dev_miou_curve.png',
    'dev_per_class_iou_curve.png',
]:
    plot_path = RUN_DIR / 'figures' / plot_name
    print(plot_path)
    display(Image(filename=str(plot_path)))


## 16. MLflow UI

Use these small cells in order: start the server, check it, open a browser link or iframe, and stop it when finished.


In [ ]:
import os
import subprocess
import time
from pathlib import Path
from google.colab import output
from IPython.display import display, HTML

DRIVE_ROOT = Path('/content/drive/MyDrive/cityscapes_robustness')
MLFLOW_DB = DRIVE_ROOT / 'mlflow.db'
MLFLOW_ARTIFACT_DIR = DRIVE_ROOT / 'mlartifacts'
LOG_DIR = DRIVE_ROOT / 'logs'

LOG_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

backend_store_uri = f'sqlite:///{MLFLOW_DB}'
artifact_root_uri = MLFLOW_ARTIFACT_DIR.resolve().as_uri()

os.environ['MLFLOW_TRACKING_URI'] = backend_store_uri
os.environ['MLFLOW_ARTIFACT_ROOT'] = artifact_root_uri

subprocess.run("pkill -f 'mlflow server' || true", shell=True, check=False)

mlflow_log = LOG_DIR / 'mlflow_ui_server.log'
cmd = [
    'mlflow',
    'server',
    '--backend-store-uri',
    backend_store_uri,
    '--default-artifact-root',
    artifact_root_uri,
    '--host',
    '0.0.0.0',
    '--port',
    '5000',
    '--allowed-hosts',
    '*',
    '--cors-allowed-origins',
    '*',
    '--x-frame-options',
    'NONE',
    '--workers',
    '1',
]

print('Starting MLflow UI:')
print(' '.join(cmd))
print('Server log:', mlflow_log)

log_file = open(mlflow_log, 'a', encoding='utf-8')
mlflow_process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
time.sleep(5)

if mlflow_process.poll() is not None:
    print('MLflow server stopped with an error. Last log lines:')
    print(mlflow_log.read_text(encoding='utf-8')[-4000:])
    raise RuntimeError('MLflow server did not start')

url = output.eval_js('google.colab.kernel.proxyPort(5000)')
print('MLflow server PID:', mlflow_process.pid)
print('MLflow UI URL:', url)
display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open MLflow UI in a new tab</a>'))
output.serve_kernel_port_as_iframe(5000, height=800)


### Check MLflow UI connection


In [ ]:
!curl -I http://127.0.0.1:5000


### Reopen MLflow UI link or iframe


In [ ]:
from google.colab import output
from IPython.display import display, HTML

url = output.eval_js('google.colab.kernel.proxyPort(5000)')
print(url)
display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open MLflow UI in a new tab</a>'))
output.serve_kernel_port_as_iframe(5000, height=800)


### Stop MLflow UI server


In [ ]:
!pkill -f "mlflow server" || true


## Resume and continue training

Resume after a runtime interruption:

1. Run sections 1-8 again.
2. In section 9, set the old `RUN_NAME`, `RESUME_TRAINING = True`, `CONTINUE_FROM_RUN = None`.
3. Run section 10. Existing `run_config.yaml` will not be overwritten.

Continue training in a new run:

1. In section 9, set a new `RUN_NAME`.
2. Set `CONTINUE_FROM_RUN = 'old_run_name'` and `RESUME_TRAINING = False`.
3. Set `MODEL_SETTINGS['unet']['training']['epochs']` to the number of new epochs to add.
4. Run section 10. The new `run_config.yaml` will store the source run, source model folder, source checkpoint, `additional_epochs`, `initial_checkpoint_epoch`, and total `training.epochs`.

Folder reminder:

- `RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME` contains results only: metrics, CSV, PNG, config, MLflow run id.
- `MODEL_DIR = DRIVE_ROOT / 'models' / RUN_NAME` contains checkpoints only: `best.pt`, `last.pt`.

Clean and darkness evaluation can be run repeatedly: every evaluation is saved separately.
